# Lesson Overview: Point Estimation

A short roadmap for the lesson ahead. **Estimation** is fundamental to statistics, and it comes in different flavors; this lesson covers **point estimation** — producing a single best number (or model) from data — through its most common method, **maximum likelihood estimation (MLE)**, its Bayesian generalization **maximum a posteriori estimation (MAP)**, and an elegant bridge between them: *MAP is MLE with regularization*. It transcribes the opening video of Lesson 6 (Point Estimation).

## The route through this lesson

1. **Maximum likelihood estimation (MLE).** The most common point-estimation method, and very popular in machine learning: pick the parameter/model under which the observed data was most probable. The notebooks ahead apply it to coins (Bernoulli), Gaussians, and the populations of Lesson 5 — recovering the sample mean, sample proportion, and (a version of) the sample variance as maximum-likelihood answers.

2. **Maximum a posteriori estimation (MAP).** MLE generalized with **Bayes' theorem** (Lesson 1 returns!): instead of maximizing the likelihood $P(\text{data} \mid \text{scenario})$ alone, maximize the *posterior* $P(\text{scenario} \mid \text{data}) \propto P(\text{data} \mid \text{scenario}) \, P(\text{scenario})$ — folding in how plausible each scenario was before seeing data.

3. **The elegant result: MAP = MLE + regularization.** The prior term acts exactly like the **regularization** penalties used in machine learning to prevent overfitting — a Gaussian prior yields L2 (ridge) regularization, a Laplace prior yields L1 (lasso). One of the nicest meeting points of probability and ML practice.

### The one-picture summary

$$
\underbrace{\arg\max_{\theta} \; P(\text{data} \mid \theta)}_{\text{MLE}}
\qquad\longrightarrow\qquad
\underbrace{\arg\max_{\theta} \; P(\text{data} \mid \theta)\,P(\theta)}_{\text{MAP}}
= \arg\max_{\theta} \; \big[\log P(\text{data} \mid \theta) + \underbrace{\log P(\theta)}_{\text{regularizer}}\big]
$$

## A thirty-second teaser

One coin, seven heads in ten flips. MLE trusts the data alone and answers $\hat{p} = 0.7$. MAP with a prior that coins are usually close to fair pulls the answer toward $\tfrac12$ — the prior *regularizes* the estimate. Everything in this lesson unpacks this picture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

heads, n = 7, 10
p = np.linspace(0.001, 0.999, 500)

likelihood = p**heads * (1 - p)**(n - heads)
prior = p**7 * (1 - p)**7                     # a 'coins are usually fair-ish' prior
posterior = likelihood * prior

fig, ax = plt.subplots(figsize=(9, 4.5))
for curve, name, color in [(likelihood, "likelihood (MLE ingredient)", "tab:blue"),
                           (prior, "prior (the regularizer)", "tab:gray"),
                           (posterior, "posterior (MAP ingredient)", "tab:green")]:
    curve = curve / np.trapezoid(curve, p)     # scale for comparability
    ax.plot(p, curve, color=color, linewidth=2, label=name)
    ax.axvline(p[np.argmax(curve)], color=color, linestyle=":", linewidth=1)

ax.set_title("7 heads in 10 flips: MLE peaks at 0.70, MAP is pulled toward 0.5")
ax.set_xlabel("coin's heads probability p")
ax.set_ylabel("scaled value")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

print(f"MLE estimate: p = {p[np.argmax(likelihood)]:.2f}")
print(f"MAP estimate: p = {p[np.argmax(posterior)]:.2f}   (prior pulls toward 0.5)")

## What to keep in mind

- **Point estimation** answers "what single value best describes the unknown?" — as opposed to interval estimation (next week's confidence intervals), which answers "what range?"
- **MLE** uses the data alone; **MAP** adds prior knowledge via Bayes' theorem; the prior plays the role of a **regularizer** that fights overfitting when data is scarce.
- With lots of data, the likelihood dominates and MLE ≈ MAP — priors matter most exactly when you can least afford to overfit.

On to the details: the next notebook builds MLE from the ground up with the popcorn-on-the-floor story.